## Resources on CIE chromaticity:
- https://www.pbr-book.org/4ed/Radiometry,_Spectra,_and_Color/Color
- https://www.itu.int/dms_pubrec/itu-r/rec/bt/R-REC-BT.2100-3-202502-I!!PDF-C.pdf
- https://registry.khronos.org/DataFormat/specs/1.3/dataformat.1.3.html#PRIMARY_CONVERSION

In [2]:
import numpy as np
from IPython.display import Math, display

np.set_printoptions(precision=9, suppress=True)


def xy_to_xyz_direction(xy):
    """Return the unscaled XYZ direction for a CIE xy chromaticity."""
    x, y = xy
    return np.array([x / y, 1.0, (1.0 - x - y) / y], dtype=np.float64)


def primaries_to_matrices(red, green, blue, white):
    """Build P, S, RGB->XYZ M, and XYZ->RGB M_inv from xy primaries."""
    r = xy_to_xyz_direction(red)
    g = xy_to_xyz_direction(green)
    b = xy_to_xyz_direction(blue)
    w = xy_to_xyz_direction(white)

    # Columns are the unscaled RGB primary directions in XYZ space.
    P = np.column_stack([r, g, b])

    # Scale the three primary directions so RGB=(1,1,1) lands on white.
    S = np.linalg.solve(P, w)
    M = P @ np.diag(S)
    M_inv = np.linalg.inv(M)
    return P, S, M, M_inv


def latex_matrix(value, precision=6):
    """Format a numpy vector or matrix as a LaTeX bmatrix."""
    a = np.asarray(value)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    rows = []
    for row in a:
        rows.append(" & ".join(f"{x:.{precision}f}" for x in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"


def show_matrix(symbol, value, precision=6):
    display(Math(fr"{symbol} = {latex_matrix(value, precision)}"))


def print_and_show_space(name, primaries):
    P, S, M, M_inv = primaries_to_matrices(**primaries)
    print(f"{name}")
    print("P =")
    print(P)
    print("S =")
    print(S)
    print("M RGB->XYZ =")
    print(M)
    print("M^-1 XYZ->RGB =")
    print(M_inv)
    print()

    label = name.replace(" ", r"\,")
    display(Math(fr"\text{{{name}}}"))
    show_matrix(fr"P_{{{label}}}", P)
    show_matrix(fr"S_{{{label}}}", S)
    show_matrix(fr"M_{{{label}\to XYZ}}", M)
    show_matrix(fr"M^{{-1}}_{{XYZ\to {label}}}", M_inv)
    return P, S, M, M_inv


BT709 = {
    "red":   (0.640, 0.330),
    "green": (0.300, 0.600),
    "blue":  (0.150, 0.060),
    "white": (0.3127, 0.3290),  # D65
}

BT2020 = {
    "red":   (0.708, 0.292),
    "green": (0.170, 0.797),
    "blue":  (0.131, 0.046),
    "white": (0.3127, 0.3290),  # D65
}

_, _, M709, M709_inv = print_and_show_space("BT.709 / sRGB", BT709)
_, _, M2020, M2020_inv = print_and_show_space("BT.2020", BT2020)

M709_to_2020 = M2020_inv @ M709
M2020_to_709 = M709_inv @ M2020

print("BT.709 / sRGB -> BT.2020 =")
print(M709_to_2020)
print()
print("BT.2020 -> BT.709 / sRGB =")
print(M2020_to_709)

show_matrix(r"M_{BT.709/sRGB\to BT.2020}", M709_to_2020)
show_matrix(r"M_{BT.2020\to BT.709/sRGB}", M2020_to_709)

# Optional sanity checks.
white = np.ones(3)
print()
print("709 white -> XYZ:", M709 @ white)
print("2020 white -> XYZ:", M2020 @ white)
print("709 white -> 2020:", M709_to_2020 @ white)

BT.709 / sRGB
P =
[[ 1.939393939  0.5          2.5        ]
 [ 1.           1.           1.         ]
 [ 0.090909091  0.166666667 13.166666667]]
S =
[0.212639006 0.715168679 0.072192315]
M RGB->XYZ =
[[0.412390799 0.357584339 0.180480788]
 [0.212639006 0.715168679 0.072192315]
 [0.019330819 0.11919478  0.950532152]]
M^-1 XYZ->RGB =
[[ 3.240969942 -1.537383178 -0.49861076 ]
 [-0.969243636  1.875967502  0.041555057]
 [ 0.05563008  -0.203976959  1.056971514]]



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

BT.709 / sRGB -> BT.2020 =
[[0.627403896 0.329283038 0.043313066]
 [0.069097289 0.919540395 0.011362316]
 [0.016391439 0.088013308 0.895595253]]

BT.2020 -> BT.709 / sRGB =
[[ 1.660491002 -0.587641139 -0.072849863]
 [-0.124550475  1.132899897 -0.008349423]
 [-0.018150763 -0.100578898  1.118729661]]


<IPython.core.display.Math object>

<IPython.core.display.Math object>


709 white -> XYZ: [0.950455927 1.          1.089057751]
2020 white -> XYZ: [0.950455927 1.          1.089057751]
709 white -> 2020: [1. 1. 1.]
